# 3. DSL definitions and operators

How GT4Py code is written: **definitions** are plain Python functions following the rules of a DSL, and **DSL operators** are the callables GT4Py builds from them. This notebook covers field operators; scan operators have their own notebook ("4. Scan operators"), and running operators on other backends is in "5. Processors and backends".

```
DSL definition --checked and wrapped by @field_operator()--> FieldOperator (embedded execution)
FieldOperator  --called with fields and scalars-->           Field (or composite values)
               (op[domain](...): the result restricted to a requested domain)
```

Contents: definitions and DSL rules · field operators · extents · composite values · requested domains.

## Definitions and DSL rules

| Concept | Kind | Meaning |
|---|---|---|
| `InspectableCallable[**P, R]` | protocol | A callable `P -> R` whose call operator's source can be retrieved with `getsource` |
| `CallableWithSource` | protocol | An `InspectableCallable` carrying its own source in `__source__`, e.g. a generated function |
| `getsource(c)` | function | Unwraps `__wrapped__`, then returns `c.__source__` or the source of `call_operator(c)`: `c` itself, or the `__call__` of its class. Extensible with `@getsource.register` |
| `DSLOperatorDefinition`, `FieldOperatorDefinition`, `ScanOperatorDefinition` | protocols | Inspectable callables whose arguments and source follow the rules of a DSL. They add no members: definitions stay plain functions |
| `DSLRules` | dataclass | What a DSL accepts: argument types, Python syntax, and `forbidden` names, and whether it is `top_level` (its calls may request a `domain`, as in programs); `violation(c)` says why `c` does not follow the rules (`None` if it does) |
| `FIELD_OPERATOR_RULES` | `DSLRules` | Fields, `int`, `float`, tuples and dataclasses as arguments; expressions, assignments, `return`, `if` (no loops); no `domain=` keyword; no `COLLECTIVES` (see "10. Local programs"); free names (globals, closure) and attributes starting with `_` are rejected, while local variables like `_k` are fine; no writes into subscripts or attributes |
| `SCAN_OPERATOR_RULES` | `DSLRules` | Scalars and tuples of them (one level of a scan); the same syntax and names |
| `is_field_operator_definition`, `is_scan_operator_definition`, `is_dsl_operator_definition` | functions | Apply the rules. They return a `TypeIs`, so type checkers narrow the callable, keeping its signature |

A callable can follow both DSLs: its kind is chosen when an operator is built from it. The rules of global programs, `PROGRAM_RULES`, are in "13. Global programs".

In [1]:
import numpy as np
from concepts import (
    FIELD_OPERATOR_RULES,
    Axis,
    CallableWithSource,
    Dims,
    Field,
    getsource,
    is_dsl_operator_definition,
    is_field_operator_definition,
    is_inspectable,
    is_scan_operator_definition,
)
from implementation import NumPyArrayField


class I(Axis): ...


class J(Axis): ...


def diffusion(u: Field[Dims[I, J], float], nu: float) -> Field[Dims[I, J], float]:
    return u + nu * (u(I + 1) + u(I - 1) + u(J + 1) + u(J - 1) - 4 * u)


def relax(previous: float, value: float) -> float:  # follows both DSLs
    return 0.5 * (previous + value)


def repeated(u: Field[Dims[I, J], float]) -> Field[Dims[I, J], float]:
    while True:  # loops are not allowed
        return u


class Damping:
    """A callable object: its definition is the source of `__call__`."""

    def __init__(self, rate: float) -> None:
        self.rate = rate

    def __call__(self, u: Field[Dims[I, J], float]) -> Field[Dims[I, J], float]:
        return (1.0 - self.rate) * u


for c in (diffusion, relax, repeated, Damping(0.1), print):
    name = getattr(c, "__name__", type(c).__name__ + "(...)")
    print(
        f"{name:12} field: {is_field_operator_definition(c)!s:5} | scan: {is_scan_operator_definition(c)!s:5} | any: {is_dsl_operator_definition(c)}"
    )
print("why not repeated:", FIELD_OPERATOR_RULES.violation(repeated))
print("top level:", FIELD_OPERATOR_RULES.top_level)  # no `domain=` in calls
print(getsource(Damping(0.1)), end="")

# A generated function has no source file: it must carry its source in `__source__`.
source = "def halve(u: Field[Dims[I, J], float]) -> Field[Dims[I, J], float]:\n    return 0.5 * u\n"
namespace = dict(globals())
exec(source, namespace)
halve = namespace["halve"]
print("halve without source:", is_field_operator_definition(halve))
halve.__source__ = source
print(
    "with __source__:", isinstance(halve, CallableWithSource), is_field_operator_definition(halve)
)


class Formula:
    """A callable built from source text: `getsource` is extended for it."""

    def __init__(self, text: str) -> None:
        self.text = text

    def __call__(self, *args):
        raise NotImplementedError


@getsource.register
def _(object: Formula) -> str:
    return object.text


formula = Formula("def double(x):\n    return 2 * x\n")
print(
    "inspectable:",
    is_inspectable(formula),
    is_inspectable(print),
    "|",
    getsource(formula).splitlines()[0],
)

diffusion    field: True  | scan: False | any: True
relax        field: True  | scan: True  | any: True
repeated     field: False | scan: False | any: False
Damping(...) field: True  | scan: False | any: True
print        field: False | scan: False | any: False
why not repeated: the syntax `While` is not accepted
top level: False
def __call__(self, u: Field[Dims[I, J], float]) -> Field[Dims[I, J], float]:
    return (1.0 - self.rate) * u
halve without source: False
with __source__: True True
inspectable: True False | def double(x):


## Field operators

`@field_operator()` checks that a function is a valid field operator definition and wraps it: it returns a `FieldOperatorWrapper`, a `FieldOperator` that runs the definition in Python when called (*embedded execution*). The wrapper is a `functools` wrapper: it has the name, docstring and `__wrapped__` of the definition, which stays untouched.

`DSLOperator[**P, R]` is the protocol of all DSL operators:

- `__gt_operator_kind__`: a `DSLOperatorKind`, `FIELD_OPERATOR` or `SCAN_OPERATOR`; a `Literal` in `FieldOperator` and `ScanOperator`, so type checkers tell them apart;
- `__gt_definition__`: the definition; `__gt_extents__`: see below;
- `__call__(*args: P.args, **kwargs: P.kwargs) -> R` and `__getitem__(domain) -> Callable[P, R]` (requested domains, below).

A `FieldOperator[P, R]` has the signature `P -> R` of its definition, and can be called from Python and from other definitions. The members are read-only properties in the protocols, so any class with such attributes is a `FieldOperator`, like the hand-written `Traced` below.

In [2]:
import dataclasses
from collections.abc import Callable
from typing import Any, ClassVar, Literal

from concepts import DomainSpec, DSLOperatorKind, Extents, FieldOperator, restrict
from implementation import field_operator, scan_operator

diffuse = field_operator()(diffusion)  # the same as decorating `diffusion`
print("operator:", type(diffuse).__name__, "| kind:", diffuse.__gt_operator_kind__)
print("wraps:", diffuse.__name__, diffuse.__wrapped__ is diffuse.__gt_definition__ is diffusion)
print("definition untouched:", vars(diffusion) == {})
# `relax` follows both DSLs: the decorator chooses the kind, without changing the function
kinds = (
    field_operator()(relax).__gt_operator_kind__,
    scan_operator(axis=J)(relax).__gt_operator_kind__,
)
print("relax as:", [kind.name for kind in kinds], "| untouched:", vars(relax) == {})

x = np.arange(5.0)
u = NumPyArrayField({I: I.at(0, 5), J: J.at(0, 5)}, (x[:, None] - 2) ** 2 + (x[None, :] - 2) ** 2)
print("diffuse(u, 0.25) =", diffuse(u, 0.25))  # forward domain: I[1:4], J[1:4]

try:
    field_operator()(repeated)
except TypeError as error:
    print("TypeError:", error)


@dataclasses.dataclass(frozen=True)
class Traced:
    """A minimal hand-written `FieldOperator`: runs its definition and reports the call."""

    __gt_operator_kind__: ClassVar[Literal[DSLOperatorKind.FIELD_OPERATOR]] = (
        DSLOperatorKind.FIELD_OPERATOR
    )
    __gt_definition__: Any
    __gt_extents__: Extents = dataclasses.field(default_factory=dict)  # simplified

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        print(f"calling {self.__gt_definition__.__name__}")
        return self.__gt_definition__(*args, **kwargs)

    def __getitem__(self, domain: DomainSpec) -> Callable[..., Any]:
        return lambda *args, **kwargs: restrict(self(*args, **kwargs), domain)


traced: FieldOperator[..., Any] = Traced(diffusion)  # checked statically, e.g. by mypy
print(traced[{I: I.at(2, 3), J: J.at(1, 4)}](u, 0.25))

operator: FieldOperatorWrapper | kind: DSLOperatorKind.FIELD_OPERATOR
wraps: diffusion True
definition untouched: True
relax as: ['FIELD_OPERATOR', 'SCAN_OPERATOR'] | untouched: True
diffuse(u, 0.25) = Field(I[1:4], J[1:4]: [[3.0, 2.0, 3.0], [2.0, 1.0, 2.0], [3.0, 2.0, 3.0]])
TypeError: 'repeated' is not a valid field operator definition: the syntax `While` is not accepted.
calling diffusion
Field(I[2:3], J[1:4]: [[2.0, 1.0, 2.0]])


## Extents

`__gt_extents__` is an `Extents`: for each argument, how the operator reads it, relative to the point being computed: a list of *reads*, each a chain of *steps* applied in order. A step is an offset, a relation (a gather), a `Boundary` (a gather with a boundary value, see "8. Chains and partial relations"), a `Reduction`, a `ScanAlong` (see "4. Scan operators"), or a `Flatten` or `Unflatten` (see "9. Footprint algebra and pairs"). Reading `u(I + 1)` is the read `((I, 1),)` and a plain `u` is `()`; on a mesh, `sum_over(e(C2E), axis=Neighbor[C2E])` reads `e` with `(C2E, Reduction(Neighbor[C2E]))` (see "2. Connectivities"). The **extent** along a dimension is the `(min, max)` range of the offsets: domains shrink by it forward and grow by it backward. Through relations and reductions, domains propagate step by step.

`implementation.py` finds the extents with a small symbolic evaluation of the definition (`read_extents`): the reads are followed through local variables, `if` branches and calls to other DSL operators, whose extents are composed with them. The extents over-approximate: both branches of an `if` contribute their reads, so the default domains of the lazy backends may be smaller than those of `eager`, and a requested domain `eager` accepts may raise on them; the values are always right.

In [3]:
from concepts import Location, Neighbor, Reduction, Relation
from implementation import sum_over


def format_step(step):
    if isinstance(step, type):  # a relation
        return step.__name__
    if isinstance(step, Reduction):
        return f"Reduction({step.axis.__name__})"
    return f"{step[0].__name__}{step[1]:+d}"


def format_reads(reads):
    return ["(" + ", ".join(map(format_step, read)) + ")" for read in reads]


@field_operator()
def upwind(u: Field[Dims[I, J], float], c: Field[Dims[I, J], float]) -> Field[Dims[I, J], float]:
    return u - 0.5 * (c + c(I - 1)) * (u - u(I - 1))  # with the speed between the points


class Cell(Location): ...


class Edge(Location): ...


class C2E(Relation[Cell, Edge], max_neighbors=3): ...


@field_operator()
def edge_excess(
    e: Field[Dims[Edge], float], c: Field[Dims[Cell], float]
) -> Field[Dims[Cell], float]:
    return sum_over(e(C2E), axis=Neighbor[C2E]) - 3 * c


for op in (diffuse, upwind, edge_excess):
    print(op.__name__, {name: format_reads(reads) for name, reads in op.__gt_extents__.items()})

diffusion {'u': ['()', '(I+1)', '(I-1)', '(J+1)', '(J-1)'], 'nu': []}
upwind {'u': ['()', '(I-1)'], 'c': ['()', '(I-1)']}
edge_excess {'e': ['(C2E, Reduction(Neighbor[C2E]))'], 'c': ['()']}


## Composite values

Besides fields and scalars, definitions take and return **composite values** (see "1. Fields and domains"): dataclass instances (`NamedCollection`s), read by attribute, and tuples of values, unpacked or built. Definitions can use conditional expressions and `if` statements, and call other DSL operators.

In [4]:
@dataclasses.dataclass
class Physics:  # a `NamedCollection`
    nu: float
    relaxation: float  # 0: none


type IJ = Field[Dims[I, J], float]


@field_operator()
def diffusion_step(u: IJ, physics: Physics) -> tuple[IJ, IJ]:
    new = diffuse(u, physics.nu)  # calls another field operator
    if physics.relaxation > 0.0:
        new = (1.0 - physics.relaxation) * new + physics.relaxation * u
    return new, new - u


@field_operator()
def swap(pair: tuple[IJ, IJ]) -> tuple[IJ, IJ]:
    first, second = pair
    return second, first


new, change = diffusion_step(u, Physics(nu=0.25, relaxation=0.0))
relaxed, _ = diffusion_step(u, Physics(nu=0.25, relaxation=0.5))
print("new     =", new)
print("change  =", change)
print("relaxed =", relaxed)
first, second = swap((u, new))
print("swap    =", first is new, second is u)

new     = Field(I[1:4], J[1:4]: [[3.0, 2.0, 3.0], [2.0, 1.0, 2.0], [3.0, 2.0, 3.0]])
change  = Field(I[1:4], J[1:4]: [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]])
relaxed = Field(I[1:4], J[1:4]: [[2.5, 1.5, 2.5], [1.5, 0.5, 1.5], [2.5, 1.5, 2.5]])
swap    = True True


## Requested domains

`op[domain](*args)` is the result of `op(*args)` restricted to a requested domain, a `DomainSpec`: one `Domain` per field of the result, with the structure of the result, or one `Domain` for all the fields. `restrict(result, domain)` does the restriction; `op(*args, domain=domain)` is a runtime shorthand that type checkers reject (a keyword cannot be added to the signature `P`).

Operators may use the requested domain to compute less: **backward domain propagation** (see "5. Processors and backends"). A domain is only requested from Python: a `domain=` keyword is not allowed in definitions. Requested domains can be index sets on unstructured dimensions. A requested domain outside the result's domain raises a `ValueError`.

In [5]:
center = {I: I.at(2, 3), J: J.at(2, 3)}
print("diffuse[center]     =", diffuse[center](u, 0.25))
print("domain= keyword     =", diffuse(u, 0.25, domain=center))
print("restrict            =", restrict(diffuse(u, 0.25), center))

row = {I: I.at(1, 2), J: J.at(1, 4)}
new, change = diffusion_step[(center, row)](u, Physics(nu=0.25, relaxation=0.0))  # one per field
print("per field           =", new, change)
new, change = diffusion_step[center](u, Physics(nu=0.25, relaxation=0.0))  # one for all
print("for all             =", new.domain == change.domain == center)
try:
    diffuse[{I: I.at(0, 2), J: J.at(1, 4)}](u, 0.25)  # the result starts at I(1)
except ValueError as error:
    print("ValueError:", error)

try:

    @field_operator()
    def inner(u: IJ) -> IJ:
        return diffuse(u, 0.25, domain=center)  # not allowed in DSL code

except TypeError as error:
    print("TypeError:", error)

diffuse[center]     = Field(I[2:3], J[2:3]: [[1.0]])
domain= keyword     = Field(I[2:3], J[2:3]: [[1.0]])
restrict            = Field(I[2:3], J[2:3]: [[1.0]])
per field           = Field(I[2:3], J[2:3]: [[1.0]]) Field(I[1:2], J[1:4]: [[1.0, 1.0, 1.0]])
for all             = True
ValueError: I[0:2] is not within the domain I[1:4], J[1:4].
TypeError: 'inner' is not a valid field operator definition: a `domain` is only given in top-level calls.
